# Anime Character Classifier (Notebook-Only)

**Purpose:** Train and evaluate an anime character classifier from face images.
**Input:** Dataset downloaded with `kagglehub` (`anuragraj03/anime-face-dataset`).
**Output:** Trained model artifacts, evaluation metrics, and single-image predictions.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "tensorflow",
    "numpy",
    "Pillow",
    "scikit-learn",
    "kagglehub",
    "opencv-python-headless",
])

import json
import random
from pathlib import Path

import cv2
import kagglehub
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split

## Runtime Setup
**Purpose:** Install dependencies and import required libraries.
**Input:** Fresh Colab runtime.
**Output:** Ready Python environment for all later cells.

In [ ]:
SEED = 42
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15

DATASET_SLUG = "anuragraj03/anime-face-dataset"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

ARTIFACTS_DIR = Path("artifacts")
MODEL_DIR = ARTIFACTS_DIR / "model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Constants configured.")

## Shared Helpers
**Purpose:** Define reusable utility functions used across the notebook.
**Input:** Constants and imported libraries.
**Output:** Functions for dataset handling, preprocessing, and model creation.

In [ ]:
def list_class_dirs(source_dir: Path):
    class_dirs = [d for d in source_dir.iterdir() if d.is_dir() and not d.name.startswith(".")]
    return sorted(class_dirs, key=lambda p: p.name.lower())


def list_images(class_dir: Path):
    return sorted([f for f in class_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTS])


def decode_and_resize(image_path, label):
    image_bytes = tf.io.read_file(image_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


def make_dataset(records, shuffle=True):
    paths = [str(path) for path, _ in records]
    labels = [label for _, label in records]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle and records:
        ds = ds.shuffle(buffer_size=len(records), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def build_model(num_classes: int):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("Shared helper functions ready.")

## Dataset Check
**Purpose:** Validate dataset location and class folders.
**Input:** Downloaded dataset directory.
**Output:** Class count and per-class image counts.

In [ ]:
dataset_path = Path(kagglehub.dataset_download(DATASET_SLUG))
SOURCE_DIR = dataset_path

class_dirs = list_class_dirs(SOURCE_DIR)
if not class_dirs:
    nested_dirs = [d for d in SOURCE_DIR.iterdir() if d.is_dir() and not d.name.startswith(".")]
    if len(nested_dirs) == 1:
        SOURCE_DIR = nested_dirs[0]
        class_dirs = list_class_dirs(SOURCE_DIR)

if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"Source directory not found: {SOURCE_DIR}")
if not class_dirs:
    raise ValueError(f"No class folders found in {SOURCE_DIR}")

summary = {d.name: len(list_images(d)) for d in class_dirs}
print(f"Source directory: {SOURCE_DIR.resolve()}")
print(f"Number of classes: {len(summary)}")
print(f"Total images: {sum(summary.values())}")
print("Images per class:")
for class_name, count in summary.items():
    print(f"  - {class_name}: {count}")

## Train/Validation/Test Split
**Purpose:** Create in-memory train/validation/test splits.
**Input:** Class directories and image paths.
**Output:** `train_records`, `val_records`, `test_records`, and split statistics.

In [ ]:
class_names = [d.name for d in class_dirs]
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

train_records, val_records, test_records = [], [], []
split_stats = {"train": {}, "val": {}, "test": {}}

for class_dir in class_dirs:
    images = list_images(class_dir)
    if len(images) < 3:
        raise ValueError(f"Class '{class_dir.name}' has less than 3 images")

    train_files, remaining = train_test_split(
        images,
        test_size=0.30,
        random_state=SEED,
        shuffle=True,
    )
    val_files, test_files = train_test_split(
        remaining,
        test_size=0.50,
        random_state=SEED,
        shuffle=True,
    )

    label = class_to_idx[class_dir.name]
    train_records.extend((path, label) for path in train_files)
    val_records.extend((path, label) for path in val_files)
    test_records.extend((path, label) for path in test_files)

    split_stats["train"][class_dir.name] = len(train_files)
    split_stats["val"][class_dir.name] = len(val_files)
    split_stats["test"][class_dir.name] = len(test_files)

print("Dataset split completed (in memory).")
for split_name, records in (("train", train_records), ("val", val_records), ("test", test_records)):
    print(f"{split_name}: {len(records)} images")

## Model Definition
**Purpose:** Build the CNN classifier architecture.
**Input:** Number of classes.
**Output:** Compiled model and saved model-summary artifact.

In [ ]:
num_classes = len(class_dirs)
model = build_model(num_classes)
model.summary()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
summary_lines = []
model.summary(print_fn=lambda x: summary_lines.append(x))
(ARTIFACTS_DIR / "model_summary.txt").write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Model summary saved to: {(ARTIFACTS_DIR / 'model_summary.txt').resolve()}")

## ANN Structure
**Purpose:** Save a plain-text architecture description.
**Input:** Current model design.
**Output:** `artifacts/ann_structure.txt`.

In [ ]:
ann_structure = (
    "ANN/CNN structure used in this project:\n\n"
    "Input: 128x128x3 RGB image\n"
    "-> Input normalization in tf.data pipeline (pixel / 255)\n"
    "-> Conv2D(32, 3x3, relu)\n"
    "-> MaxPooling2D(2x2)\n"
    "-> Conv2D(64, 3x3, relu)\n"
    "-> MaxPooling2D(2x2)\n"
    "-> Conv2D(128, 3x3, relu)\n"
    "-> MaxPooling2D(2x2)\n"
    "-> Flatten\n"
    "-> Dense(256, relu)\n"
    "-> Dropout(0.3)\n"
    "-> Dense(num_classes, softmax)\n"
)

(ARTIFACTS_DIR / "ann_structure.txt").write_text(ann_structure, encoding="utf-8")
print(ann_structure)
print(f"ANN structure saved to: {(ARTIFACTS_DIR / 'ann_structure.txt').resolve()}")

## Training
**Purpose:** Train the classifier on training data and validate each epoch.
**Input:** Train/validation datasets and compiled model.
**Output:** Training history, `best_model.keras`, and `final_model.keras`.

In [ ]:
train_ds = make_dataset(train_records, shuffle=True)
val_ds = make_dataset(val_records, shuffle=False)

(MODEL_DIR / "class_names.json").write_text(
    json.dumps(class_names, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

model = build_model(num_classes=len(class_names))
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best_model.keras"),
        monitor="val_accuracy",
        save_best_only=True,
    ),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
model.save(MODEL_DIR / "final_model.keras")
print(f"Training complete. Final model saved to: {(MODEL_DIR / 'final_model.keras').resolve()}")
print(f"Best model saved to: {(MODEL_DIR / 'best_model.keras').resolve()}")
print("Final epoch metrics:", {k: float(v[-1]) for k, v in history.history.items()})

## Evaluation
**Purpose:** Measure trained-model performance on unseen test data.
**Input:** Test dataset and saved model checkpoint.
**Output:** Test loss and accuracy metrics.

In [ ]:
test_ds = make_dataset(test_records, shuffle=False)

best_model_path = MODEL_DIR / "best_model.keras"
model_path = best_model_path if best_model_path.exists() else (MODEL_DIR / "final_model.keras")
trained_model = tf.keras.models.load_model(model_path)

loss, accuracy = trained_model.evaluate(test_ds, verbose=1)
print(f"Loaded model: {model_path.resolve()}")
print(f"Classes ({len(class_names)}): {class_names}")
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

## Single Image Prediction
**Purpose:** Predict one image after optional face detection/cropping.
**Input:** Uploaded image or local image path.
**Output:** Predicted class, confidence, and top candidate scores.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_path = list(uploaded.keys())[0]
except Exception:
    uploaded_path = input("Enter image path to test: ").strip()

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_cascade.empty():
    raise RuntimeError("Failed to load Haar face detector.")

original_img = Image.open(uploaded_path).convert("RGB")
rgb = np.array(original_img)
gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48))

if len(faces) == 0:
    face_img = original_img
    face_bbox = None
else:
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])
    pad = int(0.2 * max(w, h))
    x0 = max(0, x - pad)
    y0 = max(0, y - pad)
    x1 = min(rgb.shape[1], x + w + pad)
    y1 = min(rgb.shape[0], y + h + pad)
    face_img = Image.fromarray(rgb[y0:y1, x0:x1])
    face_bbox = (x0, y0, x1, y1)

model_img = face_img.resize((IMG_SIZE, IMG_SIZE))
arr = np.expand_dims(np.array(model_img, dtype=np.float32) / 255.0, axis=0)

class_names = json.loads((MODEL_DIR / "class_names.json").read_text(encoding="utf-8"))
probs = trained_model.predict(arr, verbose=0)[0]
pred_idx = int(np.argmax(probs))
top_indices = np.argsort(probs)[::-1][: min(3, len(probs))]

print(f"Image: {uploaded_path}")
if face_bbox is None:
    print("Face detection: no face found, used full image.")
else:
    print(f"Face detection: used crop bbox={face_bbox}")
print(f"Predicted character: {class_names[pred_idx]}")
print(f"Confidence: {float(probs[pred_idx]):.6f}")
print("Top predictions:")
for idx in top_indices:
    print(f"  - {class_names[int(idx)]}: {float(probs[int(idx)]):.6f}")

## Save Weights
**Purpose:** Persist model weights for later reuse.
**Input:** Trained model in memory.
**Output:** `artifacts/model_weights.weights.h5`.

In [ ]:
weights_path = ARTIFACTS_DIR / "model_weights.weights.h5"
weights_path.parent.mkdir(parents=True, exist_ok=True)
trained_model.save_weights(weights_path)
print(f"Weights saved to: {weights_path.resolve()}")